# 6. Forks

A **fork** is your own copy of someone else's repository, on the server.
You make one when you want to contribute to a project you are not allowed
to push to.

```mermaid
flowchart LR
    up["upstream (the group's repository)"] -->|fork| fork["origin (your fork)"]
    fork -->|git clone| mine["your laptop"]
    mine -->|git push| fork
    fork -->|pull request| up
    up -->|git fetch upstream| mine
```

Get updates from **upstream**, push your work to **origin**.

In [1]:
!rm -rf ~/git-course/06-forks
!mkdir -p ~/git-course/06-forks
%cd ~/git-course/06-forks

/home/emircem/git-course/06-forks


## The group's project

Prof. Kaya's repository, on the server as `upstream.git`. You cannot push
to it.

In [2]:
!git init --bare upstream.git
!mkdir kaya
%cd kaya

Initialized empty Git repository in /home/emircem/git-course/06-forks/upstream.git/
/home/emircem/git-course/06-forks/kaya


In [3]:
%%writefile hydrostatics.py
"""Hydrostatics for a box-shaped barge."""

RHO = 1025.0  # kg/m^3, sea water


def volume(length, beam, draft):
    """Immersed volume of the hull, in m^3."""
    return length * beam * draft


def displacement(length, beam, draft):
    """Mass of the water the hull pushes aside, in kg."""
    return RHO * volume(length, beam, draft)

Writing hydrostatics.py


In [4]:
!git init
!git config user.name "Prof. Kaya"
!git add hydrostatics.py
!git commit -m "Add volume and displacement for a box barge"
!git remote add origin ../upstream.git
!git push -u origin main

Initialized empty Git repository in /home/emircem/git-course/06-forks/kaya/.git/
[main (root-commit) 901c08b] Add volume and displacement for a box barge
 1 file changed, 13 insertions(+)
 create mode 100644 hydrostatics.py
Enumerating objects: 3, done.
Counting objects: 100% (3/3), done.
Delta compression using up to 20 threads
Compressing objects: 100% (2/2), done.
Writing objects: 100% (3/3), 439 bytes | 439.00 KiB/s, done.
Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
To ../upstream.git
 * [new branch]      main -> main
branch 'main' set up to track 'origin/main'.


## Forking

On GitHub you press **Fork**. Behind the button, the server simply copies
the repository:

In [5]:
%cd ..
!git clone --bare upstream.git fork.git

/home/emircem/git-course/06-forks
Cloning into bare repository 'fork.git'...
done.


## Cloning your fork

Clone **your fork**, so that `origin` is a place you can push to:

In [6]:
!git clone fork.git mine
%cd mine
!git remote -v

Cloning into 'mine'...
done.
/home/emircem/git-course/06-forks/mine
origin	/home/emircem/git-course/06-forks/fork.git (fetch)
origin	/home/emircem/git-course/06-forks/fork.git (push)


## Adding upstream

Your fork does not update itself when the group keeps working. Add the
original as a second remote, called `upstream`:

In [7]:
!git remote add upstream ../upstream.git
!git remote -v

origin	/home/emircem/git-course/06-forks/fork.git (fetch)
origin	/home/emircem/git-course/06-forks/fork.git (push)
upstream	../upstream.git (fetch)
upstream	../upstream.git (push)


## Doing the work

Always on a branch:

In [8]:
!git switch -c block-coefficients

Switched to a new branch 'block-coefficients'


In [9]:
%%writefile -a hydrostatics.py


TYPICAL_CB = {
    "pontoon": 1.00,
    "tanker": 0.85,
    "container ship": 0.65,
    "patrol boat": 0.50,
}

Appending to hydrostatics.py


In [10]:
!git add hydrostatics.py
!git commit -m "Add a table of typical block coefficients"

[block-coefficients f47b597] Add a table of typical block coefficients
 1 file changed, 8 insertions(+)


Push the branch to **your fork**:

In [11]:
!git push -u origin block-coefficients

Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 20 threads
Compressing objects: 100% (2/2), done.
Writing objects: 100% (3/3), 389 bytes | 389.00 KiB/s, done.
Total 3 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
To /home/emircem/git-course/06-forks/fork.git
 * [new branch]      block-coefficients -> block-coefficients
branch 'block-coefficients' set up to track 'origin/block-coefficients'.


## The pull request

On GitHub you now click **Compare & pull request**. A pull request is a
message: "please merge my branch into your project".

Here is what Prof. Kaya does to accept it. First, fetch your branch from
your fork:

In [12]:
%cd ../kaya
!git remote add contributor ../fork.git
!git fetch contributor

/home/emircem/git-course/06-forks/kaya
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 3 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 369 bytes | 369.00 KiB/s, done.
From ../fork
 * [new branch]      block-coefficients -> contributor/block-coefficients
 * [new branch]      main               -> contributor/main


Look at what changed:

In [13]:
!git diff main contributor/block-coefficients

diff --git a/hydrostatics.py b/hydrostatics.py
index f3c3cd6..c215aa5 100644
--- a/hydrostatics.py
+++ b/hydrostatics.py
@@ -11,3 +11,11 @@ def volume(length, beam, draft):
 def displacement(length, beam, draft):
     """Mass of the water the hull pushes aside, in kg."""
     return RHO * volume(length, beam, draft)
+
+
+TYPICAL_CB = {
+    "pontoon": 1.00,
+    "tanker": 0.85,
+    "container ship": 0.65,
+    "patrol boat": 0.50,
+}


Looks good, so merge it and push:

In [14]:
!git merge contributor/block-coefficients -m "Merge pull request #1: block coefficients"
!git push

Updating 901c08b..f47b597
Fast-forward (no commit created; -m option ignored)
 hydrostatics.py | 8 ++++++++
 1 file changed, 8 insertions(+)
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 20 threads
Compressing objects: 100% (2/2), done.
Writing objects: 100% (3/3), 389 bytes | 389.00 KiB/s, done.
Total 3 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
To ../upstream.git
   901c08b..f47b597  main -> main


Meanwhile, Prof. Kaya also adds a README:

In [15]:
%%writefile README.md
# Barge hydrostatics

Writing README.md


In [16]:
!git add README.md
!git commit -m "Add a README"
!git push

[main 5e11cd7] Add a README
 1 file changed, 1 insertion(+)
 create mode 100644 README.md
Enumerating objects: 4, done.
Counting objects: 100% (4/4), done.
Delta compression using up to 20 threads
Compressing objects: 100% (2/2), done.
Writing objects: 100% (3/3), 307 bytes | 307.00 KiB/s, done.
Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
To ../upstream.git
   f47b597..5e11cd7  main -> main


## Keeping your fork up to date

Your fork and your laptop know nothing about any of this yet. Fetch from
**upstream**:

In [17]:
%cd ../mine
!git switch main
!git fetch upstream

/home/emircem/git-course/06-forks/mine
Switched to branch 'main'
Your branch is up to date with 'origin/main'.
remote: Enumerating objects: 4, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 287 bytes | 287.00 KiB/s, done.
From ../upstream
 * [new branch]      main       -> upstream/main


In [18]:
!git log --oneline --all

5e11cd7 (upstream/main, upstream/HEAD) Add a README
f47b597 (origin/block-coefficients, block-coefficients) Add a table of typical block coefficients
901c08b (HEAD -> main, origin/main, origin/HEAD) Add volume and displacement for a box barge


Merge the group's work into your `main`, and push it to your fork:

In [19]:
!git merge upstream/main
!git push origin main

Updating 901c08b..5e11cd7
Fast-forward
 README.md       | 1 +
 hydrostatics.py | 8 ++++++++
 2 files changed, 9 insertions(+)
 create mode 100644 README.md
Enumerating objects: 4, done.
Counting objects: 100% (4/4), done.
Delta compression using up to 20 threads
Compressing objects: 100% (2/2), done.
Writing objects: 100% (3/3), 307 bytes | 307.00 KiB/s, done.
Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
To /home/emircem/git-course/06-forks/fork.git
   901c08b..5e11cd7  main -> main


In [20]:
!ls

README.md  hydrostatics.py


Your work came back through the group's repository, together with the
README. You will repeat these steps for as long as you contribute:

```
git switch main
git fetch upstream
git merge upstream/main
git push origin main
```

## Fork, clone, or branch?

| | What it is | When |
| - | - | - |
| Branch | a line of work inside one repository | for every piece of work |
| Clone | a copy on your computer | to work on a project |
| Fork | your copy on the server | when you cannot push to the original |

If you *can* push to a project (your own, your group's), you don't need a
fork: just push a branch and open a pull request.

## Exercises

1. Make a second contribution on a new branch, and accept it as Prof. Kaya.
2. Run `git remote -v` in `mine` and say what each remote is for.

Next: [7. Cherry-pick](07-cherry-pick.ipynb)